In [22]:
import pandas as pd

In [23]:
def load_ibge_harvest(csv, feature_name):

    years_to_analize = [str(year) for year in range(2003, 2025)]
    cols = ['level', 'code', 'name'] + years_to_analize
    
    df_raw = pd.read_csv(csv, header=None, names=cols, sep=';', engine='python')
    
    df_mu = df_raw[df_raw['level'] == 'MU']

    df_mu = df_mu.replace('-', 0).replace('...', 0)

    df_mu_pv = pd.melt(
        df_mu,
        id_vars=['code', 'name'],
        value_vars=years_to_analize,
        var_name='year',
        value_name=feature_name
    )
    
    df_mu_pv['year'] = df_mu_pv['year'].astype(int)
    df_mu_pv[feature_name] = df_mu_pv[feature_name].astype(int)
    
    return df_mu_pv

### Reading the yeld and planted area for the municipalities, leaving only those that have data for all years

In [24]:
df_area = load_ibge_harvest('ibge/area_plantada.csv', 'planted_area_ha')
df_yield = load_ibge_harvest('ibge/kg_por_hectare.csv', 'yield_kg_ha')
df_yield.drop(columns=['name'], inplace=True)

df_ibge_combined = pd.merge(
    df_area,
    df_yield,
    on=['code', 'year'],
    how='inner'
)

df_ibge_combined

df_ibge_combined = df_ibge_combined.loc[:, ~df_ibge_combined.columns.str.endswith('_duplicado')]

cidades_com_zero = df_ibge_combined[df_ibge_combined['yield_kg_ha'] == 0]['name'].unique()

# 3. Filtra o dataset mantendo apenas as cidades que NÃO estão na lista acima
df_ibge_combined = df_ibge_combined[~df_ibge_combined['name'].isin(cidades_com_zero)]

df_ibge_combined.to_csv('ibge/df_ibge_combined.csv', index=False)

In [25]:
cities_yeld = df_ibge_combined[['code', 'name']].drop_duplicates()
ibge_coords = pd.read_csv('ibge/cities_coords.csv', sep=',')

cities_coords = pd.merge(
    cities_yeld,
    ibge_coords[['codigo_ibge', 'latitude', 'longitude']],
    left_on='code',
    right_on='codigo_ibge',
    how='left'
)

cities_coords['name'] = cities_coords['name'].str.strip()

cities_coords.drop(columns=['codigo_ibge']).drop_duplicates().to_csv('ibge/cities_to_analize.csv', index=False)

In [26]:
df_ibge_final = df_ibge_combined

In [27]:
%store df_ibge_final

Stored 'df_ibge_final' (DataFrame)


In [28]:
len(df_ibge_final)

946